# Proposed methods on Kaggle: learned per-task head mixing + task-specific reading of the judgment

**What this runs** (`PLAN = "mix"`, both T4 GPUs, about 6-7 h per GPU):
* **(A) Learned per-task mixing** (`--mix_heads dur,typ,disp`): every court keeps the shared head *and* its own head for each task and learns how much to trust each (one weight per court and task).
* **(B) Task-specific reading** (`--segments 2`): the start and the end of each judgment are encoded separately; each task head learns its own weighting of the two.
* **(C) A + B together.** Every federated run also reports its per-court fine-tuned variant (`+ft`).
* Seeds 0, 1, 2 (all seed-0 runs first), then local and centralized with (B) to separate the architecture gain from the federated gain.

**Before running**
1. **Add Input** → the dataset with `judgments_labelled.csv` and the new scripts (`judicial_fl_mix_dataset.zip`: `fl_judicial.py`, `run_queue.py`, `summarize_results.py` + the sweep's `runs/`).
2. Also **Add Input** → *Your Work* → the output of your **latest** notebook run (mini / novel / extra runs), so all earlier results are merged into one table. Optional but recommended.
3. Settings: **GPU T4 x2**, **Internet On**. Then **Save Version → Save & Run All (Commit)**.

Finished runs are skipped; an interrupted FL run resumes from its last round, so if the 12 h limit or the quota cuts the session, attach this notebook's output and run it again.

Outputs: `runs/all_results.csv`, `comparison.csv`, `proposed_methods.csv`, `mixing_weights_summary.csv`, `segment_weights_summary.csv`.

In [ ]:
!pip install -q peft
!pip uninstall -y -q torchao
import os, glob, shutil, subprocess, sys, time, torch
MARK = {"fl_judicial.py": "mix_heads", "run_queue.py": '"mix"', "summarize_results.py": "sel_val_loss"}
def find(name):
    """newest copy: if several versions are attached, take the one with the current features"""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    good = [h for h in hits if MARK.get(name, "") in open(h).read()]
    return (good or hits or [None])[0]
# the CSV may have been uploaded under a slightly different name (e.g. judgments_labelled_1.csv)
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
cands = [p for p in csvs if "judgments_labelled" in os.path.basename(p)]
DATA = cands[0] if cands else None
assert DATA, f"judgments_labelled*.csv not found under /kaggle/input. CSVs found: {csvs}"
for f in ["fl_judicial.py", "run_queue.py", "summarize_results.py"]:
    p = find(f); assert p, f"{f} not found under /kaggle/input: upload it to your dataset"
    shutil.copy(p, f"/kaggle/working/{f}")
os.chdir("/kaggle/working")
N_GPU = torch.cuda.device_count()
print("data:", DATA, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert "mix_heads" in open("fl_judicial.py").read(), "old fl_judicial.py in the dataset: upload the new one"
assert '"mix"' in open("run_queue.py").read(), "old run_queue.py in the dataset (no 'mix' plan): upload the new one"
assert N_GPU >= 1, "no GPU: set Accelerator = GPU T4 x2"
import peft, urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); print("internet: ok")
except Exception as e:
    raise RuntimeError(f"no internet ({e}): turn Internet On (phone-verified account needed)")

In [ ]:
# Resume: restore and MERGE runs/ from every attached input (dataset zip and/or earlier notebook outputs)
import zipfile, pandas as pd
os.makedirs("runs", exist_ok=True)
for i, z in enumerate(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
    with zipfile.ZipFile(z) as f:
        if any(n.endswith("all_results.csv") for n in f.namelist()):
            f.extractall(f"/kaggle/working/_restore/{i}"); print("unzipped", z)
srcs = glob.glob("/kaggle/input/**/all_results.csv", recursive=True) + \
       glob.glob("/kaggle/working/_restore/**/all_results.csv", recursive=True)
frames = [pd.read_csv("runs/all_results.csv")] if os.path.exists("runs/all_results.csv") else []
for p in srcs:
    d = os.path.dirname(p)
    for sub in os.listdir(d):
        if os.path.isdir(os.path.join(d, sub)):
            shutil.copytree(os.path.join(d, sub), os.path.join("runs", sub), dirs_exist_ok=True)
    for f in glob.glob(os.path.join(d, "tok_cache_*.pkl")): shutil.copy(f, "runs/")
    frames.append(pd.read_csv(p)); print("restored", d)
if frames:   # union of all sources; a run present in several inputs is kept once
    pd.concat(frames, ignore_index=True).drop_duplicates(subset=["run", "method", "court"], keep="last") \
      .to_csv("runs/all_results.csv", index=False)
done = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/per_court_test.csv")]
partial = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/ckpt.pt") and d not in done]
print(f"finished runs: {len(done)} | interrupted runs that will resume: {partial}")
assert os.path.exists("runs/all_results.csv"), "no previous results found: check the attached inputs"

In [ ]:
PLAN = "mix"             # "mix" (proposed methods) | "novel" | "mini" | "sweep" | "confirm"
BUDGET_HOURS = 10.5       # queues stop launching new runs after this; Kaggle hard limit is 12 h
ENCODERS = ["roberta-base", "law-ai/InLegalBERT"]

def launch(gpu, plan, extra, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, "-u", "run_queue.py", "--data", DATA, "--plan", plan, "--budget_hours", str(BUDGET_HOURS), *extra]
    return subprocess.Popen(cmd, env=env, stdout=open(log, "a"), stderr=subprocess.STDOUT)

t0 = time.time()
if PLAN in ("mix", "confirm") and N_GPU >= 2:
    procs = [launch(i, PLAN, ["--shard", str(i), "--n_shards", "2"], f"queue_gpu{i}.log") for i in range(2)]
elif PLAN == "sweep" and N_GPU >= 2:
    procs = [launch(i, "sweep", ["--encoder", e], f"queue_gpu{i}.log") for i, e in enumerate(ENCODERS[:2])]
elif PLAN == "mini" and N_GPU >= 2:
    procs = [launch(i, "mini", ["--seeds", str(i + 1)], f"queue_gpu{i}.log") for i in range(2)]
else:   # one GPU: the whole queue on GPU 0 (whatever does not fit continues next session)
    extra = {"sweep": ["--encoder", ENCODERS[1]], "mini": ["--seeds", "1,2"]}.get(PLAN, ["--shard", "0", "--n_shards", "1"])
    procs = [launch(0, PLAN, extra, "queue_gpu0.log")]
print("launched", len(procs), "queue(s):", PLAN)

In [ ]:
# Monitor until the queues finish (prints progress every 10 minutes)
import re
while any(p.poll() is None for p in procs):
    time.sleep(600)
    print(f"\n===== {(time.time()-t0)/3600:.2f} h =====")
    for log in sorted(glob.glob("queue_gpu*.log")):
        lines = [l for l in open(log).read().splitlines() if re.search(r"shard|start|mean:|chosen|choice|least VALID|stage|early stop|best round|Error|skipped", l)]
        print(log, "\n  " + "\n  ".join(lines[-6:]))
print("all queues finished")
for log in sorted(glob.glob("queue_gpu*.log")):
    print(open(log).read()[-3000:])
# fail loudly if any run crashed (otherwise the summary silently shows only old results)
errs = []
for log in glob.glob("logs/*.log") + glob.glob("queue_gpu*.log"):
    txt = open(log).read()
    if "Traceback" in txt or "Error" in txt:
        errs.append(log); print(f"\n!!!!! errors in {log}:\n" + txt[-2500:])
assert not errs, f"runs failed, see logs above: {errs}"

In [ ]:
!python summarize_results.py runs/all_results.csv

In [ ]:
# Proposed methods: summary (mean +/- std over seeds), learned mixing weights, segment attention
import pandas as pd, numpy as np, json
r = pd.read_csv("runs/all_results.csv")
cfg = {d: json.load(open(f"runs/{d}/config.json")) for d in r.run.unique() if os.path.exists(f"runs/{d}/config.json")}
get = lambda run, k, dflt: cfg.get(run, {}).get(k, dflt) or dflt
r["mix"] = r.run.map(lambda x: get(x, "mix_heads", "none")); r["segments"] = r.run.map(lambda x: get(x, "segments", 1))
r["local_heads"] = r.run.map(lambda x: get(x, "local_heads", "none")); r["algo"] = r.run.map(lambda x: get(x, "algo", ""))
new = r[(r.mix != "none") | (r.segments == 2)]
if new.empty: print("no proposed-method runs finished yet")
else:
    enc, lr = new.model.iloc[0], new.lr.iloc[0]
    sv = new[new.segments == 1]                       # truncation used by the single-view runs
    base_trunc = sv.trunc.iloc[0] if len(sv) else "headtail"
    sel = r[(r.model == enc) & (r.lr == lr) & (r.max_len == new.max_len.iloc[0]) & (r.local_heads == "none")
            & (r.method.isin(["local_only", "centralized", "fedavg", "fedavg+ft"]))
            & ((r.algo == "fedavg") | r.method.isin(["local_only", "centralized"]))
            & ((r.trunc == base_trunc) | (r.segments == 2))]
    sel = sel.assign(variant=sel.method + np.where(sel.mix != "none", " | mix", "") + np.where(sel.segments == 2, " | seg2", ""))
    per_seed = sel.groupby(["variant", "seed"]).agg(val_loss=("sel_val_loss", "mean"), MAE_log=("MAE_log", "mean"),
        Spearman=("Spearman", "mean"), disp_F1=("disp_F1", "mean"), type_F1=("type_F1", "mean"),
        worst_MAE_log=("MAE_log", "max"), worst_disp_F1=("disp_F1", "min"))
    g = per_seed.groupby("variant"); m, sd, n = g.mean(), g.std().fillna(0), g.size().rename("seeds")
    tab = (m.round(3).astype(str) + " ± " + sd.round(3).astype(str)).join(n).loc[m.MAE_log.sort_values().index]
    tab.to_csv("proposed_methods.csv"); print(f"{enc}, lr={lr}: test set, mean over courts, mean ± std over seeds"); display(tab)

    mw = [pd.read_csv(f).assign(run=os.path.basename(os.path.dirname(f))) for f in glob.glob("runs/*/mixing_weights.csv")]
    if mw:
        mw = pd.concat(mw); mw["arch"] = np.where(mw.run.str.contains("_seg2"), "seg2", "single view")
        mws = mw.groupby(["arch", "court"])[[c for c in mw.columns if c.startswith("mix_")]].agg(["mean", "std"]).round(3)
        mws.to_csv("mixing_weights_summary.csv")
        print("\nLearned weight on the SHARED head per court and task (1 = shared only, 0 = own head only), over seeds:")
        display(mws)
    sw = r[(r.segments == 2) & r.method.isin(["fedavg", "local_only", "centralized"])]
    cols = [c for c in ["seg_end_dur", "seg_end_typ", "seg_end_disp"] if c in sw]
    if len(sw) and cols:
        sws = sw.groupby(["method", "court"])[cols].mean().round(3); sws.to_csv("segment_weights_summary.csv")
        print("\nAttention on the END of the judgment, per task (0.5 = equal; >0.5 reads the end more):"); display(sws)

In [ ]:
# Per-court view of seed-0 runs at the chosen setting (best FedAvg on validation)
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv("runs/all_results.csv"); r = r[(r.seed == 0) & (r.max_len == 256)]
fa = r[r.method == "fedavg"].groupby(["model", "lr", "trunc"]).sel_val_loss.mean()
for (enc, lr, tr) in fa.sort_values().index[:2]:
    s = r[(r.model == enc) & (r.lr == lr) & (r.trunc == tr)]
    s = s.assign(label=s.method + " | " + s.run.str.extract(r"(mu[\d.]+|beta-?[\d.]+|slr[\d.e-]+|m0\.5)", expand=False).fillna(""))
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    for a, m in zip(ax, ["MAE_log", "disp_F1"]):
        s.pivot_table(index="court", columns="label", values=m).plot.bar(ax=a, legend=(m == "disp_F1"))
        a.set_title(f"{m} per court — {enc}, lr={lr}, {tr}"); a.set_xlabel("")
    ax[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(f"per_court_{enc.split('/')[-1]}.png", dpi=130); plt.show()